## Repaso

In [295]:
import csv
from collections import defaultdict
import copy
from prettytable import PrettyTable

**Ejercicio 7 (Nivel 4): reto analítico**

**Objetivo**: con los datos limpios, comparar el desempeño de enero frente a febrero.\
**Resultado requerido**: una tabla simple impresa (país, total enero, total febrero) y una interpretación escrita de 4 a 5 frases que incluya:
- Qué patrón ves.
- Qué limitaciones tienen estos datos, incluyendo el efecto de las filas que descartaste.
- Qué decisión de negocio podría apoyar.

**Restricción**: decide tú cómo obtener el mes a partir de la fecha y justifica tu decisión en una línea.

In [296]:
# Leer archivo csv
with open("data/ventas_sucias.csv", encoding="utf-8", newline="") as data_file:
    reader = csv.DictReader(data_file)
    data = list(reader)

In [297]:
# Crear una copia de la data orginal para trabajar sobre la copia.
data_copy = copy.deepcopy(data)

In [298]:
# Corregir nombres de países que tengan espacios en blanco y no tengan primer letra en mayúscula
for row in data_copy:
    row['pais'] = row['pais'].title().strip()

In [299]:
unrecoverable_rows = []

# Convertir montos recuperables a decimal. Si el monto tiene símbolo $, se elimina el símbolo y se convierte.
for row in data_copy[:]:
    try:
        row['monto'] = float(row['monto'].replace('$', '').strip())
    except ValueError:
        unrecoverable_rows.append(row)

In [300]:
print(f"Unrecoverable rows:")
for row in unrecoverable_rows:
    print(row)

Unrecoverable rows:
{'id_venta': '5', 'cliente': 'Sofía', 'pais': 'México', 'fecha': '2026-01-15', 'monto': ''}
{'id_venta': '9', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-02-14', 'monto': 'abc'}


In [ ]:
# Agregar columna mes al dataset para poder usar está columna como guía para la sumatoria de totales por mes de cada país.
# Los meses quedan identificados por un string con el número de mes.
for row in data_copy:
    row['mes'] = row['fecha'].split('-')[1]
    print(row)

{'id_venta': '1', 'cliente': 'Ana', 'pais': 'Guatemala', 'fecha': '2026-01-05', 'monto': 120.5, 'mes': '01'}
{'id_venta': '2', 'cliente': 'Luis', 'pais': 'México', 'fecha': '2026-01-07', 'monto': 80.0, 'mes': '01'}
{'id_venta': '3', 'cliente': 'Marta', 'pais': 'Guatemala', 'fecha': '2026-01-09', 'monto': 45.25, 'mes': '01'}
{'id_venta': '4', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-01-12', 'monto': 200.0, 'mes': '01'}
{'id_venta': '5', 'cliente': 'Sofía', 'pais': 'México', 'fecha': '2026-01-15', 'monto': '', 'mes': '01'}
{'id_venta': '6', 'cliente': 'Diego', 'pais': 'Guatemala', 'fecha': '2026-01-20', 'monto': 99.99, 'mes': '01'}
{'id_venta': '7', 'cliente': 'Ana', 'pais': 'Guatemala', 'fecha': '2026-02-02', 'monto': 75.0, 'mes': '02'}
{'id_venta': '8', 'cliente': 'Luis', 'pais': 'México', 'fecha': '2026-02-10', 'monto': 130.0, 'mes': '02'}
{'id_venta': '9', 'cliente': 'Pedro', 'pais': 'Colombia', 'fecha': '2026-02-14', 'monto': 'abc', 'mes': '02'}
{'id_venta': '10', 'cli

In [302]:
# Verificar los meses y paises presentes en el dataset
months = set([row['mes'] for row in data_copy])
countries = set([row['pais'] for row in data_copy])
print(months)
print(countries)

{'02', '01'}
{'México', 'Colombia', 'Guatemala'}


In [303]:
# Descartar filas irrecuperables por montos inválidos
for row in data_copy:
    if not isinstance(row['monto'], float):
        data_copy.remove(row)

In [304]:
jan_totals = defaultdict(float)
feb_totals = defaultdict(float)

# Calcular totales por país para enero y febrero
for row in data_copy:
    if row['mes'] == '01':
        jan_totals[row['pais']] += row['monto']
    else:
        feb_totals[row['pais']] += row['monto']

jan_totals = dict(jan_totals)
feb_totals = dict(feb_totals)

In [305]:
# Encontrar si hay un país sin total calculado para agregarlo a los diccionarios de totales
for country in countries:
    if country not in jan_totals:
        jan_totals[country] = None
    if country not in feb_totals:
        feb_totals[country] = None

In [306]:
# Resultados
table = PrettyTable()

table.add_column('pais', ['Guatemala', 'México', 'Colombia'])
table.add_column('total enero', [sum for sum in jan_totals.values()])
table.add_column('total febrero', [sum for sum in feb_totals.values()])
print(table)

+-----------+-------------+---------------+
|    pais   | total enero | total febrero |
+-----------+-------------+---------------+
| Guatemala |    265.74   |     135.0     |
|   México  |     80.0    |     130.0     |
|  Colombia |    200.0    |      None     |
+-----------+-------------+---------------+
